# Pipeline complet — Annotation v4 + Corrections + Analyses
## Ré-annotation par période pivot, Fix A1/B4, NLP complémentaires, visualisations

**Ordre d'exécution :**
1. Chargement des données
2. Fix A1 (score lexical sans circularité)
3. Fix B4 (panel fixe pour polarisation)
4. Pipeline d'annotation v4 (7 batchs)
5. Post-processing et comparaison v3/v4
6. Analyses NLP : Fightin' Words temporel, FEEL, contagion cessez-le-feu
7. Visualisations de synthèse

## 1. Configuration et chargement

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# S'assurer d'être dans projet_gaza
if not (Path.cwd() / "data").exists():
    _parent = Path.cwd().parent
    if (_parent / "data").exists():
        os.chdir(_parent)

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr, kendalltau

# Classification politique
BLOCS = {
    "Gauche radicale": ["LFI-NFP", "LFI", "GDR"],
    "Gauche moderee": ["SOC", "PS-NFP", "ECO", "ECO-NFP"],
    "Centre / Majorite": ["REN", "MODEM", "HOR", "EPR", "DEM"],
    "Droite": ["LR", "RN", "UDR", "NI"],
}
GROUP_TO_BLOC = {g: b for b, gs in BLOCS.items() for g in gs}
BLOC_ORDER = list(BLOCS.keys())

OCT7 = pd.Timestamp("2023-10-07")
DATA_DIR = Path("data/annotated/predictions")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)

print("Configuration OK. DATA_DIR:", DATA_DIR.absolute())

In [ ]:
# Chargement des parquets
tw_raw = pd.read_parquet(DATA_DIR / "tweets_v3_full_clean.parquet")
iv_raw = pd.read_parquet(DATA_DIR / "interventions_v3_full_clean.parquet")
print(f"Tweets : {len(tw_raw):,}")
print(f"Interventions : {len(iv_raw):,}")

In [ ]:
# Harmonisation et filtrage (confidence >= 0.70, pas off-topic)
tw = tw_raw.copy()
tw["author"] = tw["depute_name"].fillna(tw["username"])
tw["group"] = tw["groupe_politique"].fillna("UNKNOWN")
tw["bloc"] = tw["group"].map(GROUP_TO_BLOC).fillna("UNKNOWN")
tw["date"] = pd.to_datetime(tw["date_parsed"], errors="coerce")
tw["text_clean"] = tw["text"]
tw["arena"] = "Twitter"

iv = iv_raw.copy()
iv["author"] = iv.get("speaker_name", iv.get("matched_name", iv.get("AUTEUR", "")))
iv["group"] = iv["GROUPE"].fillna(iv.get("matched_group", "UNKNOWN"))
iv["bloc"] = iv["group"].map(GROUP_TO_BLOC).fillna("UNKNOWN")
iv["date"] = pd.to_datetime(iv["sitting_date"], errors="coerce")
iv["text_clean"] = iv["cleaned_text"].fillna(iv.get("TEXTE", ""))
iv["arena"] = "AN"

shared_cols = ["author", "group", "bloc", "date", "text_clean", "stance_v3", "confidence_v3",
               "primary_frame_v3", "primary_target_v3", "is_off_topic_v3", "arena", "retweets", "likes"]
for c in shared_cols:
    if c not in tw.columns: tw[c] = np.nan
    if c not in iv.columns: iv[c] = 0 if c in ["retweets", "likes"] else np.nan

tw_f = tw[(~tw["is_off_topic_v3"].fillna(False)) & (tw["confidence_v3"] >= 0.70) & (tw["bloc"] != "UNKNOWN") & (tw["date"].notna())]
iv_f = iv[(~iv["is_off_topic_v3"].fillna(False)) & (iv["confidence_v3"] >= 0.70) & (iv["bloc"] != "UNKNOWN") & (iv["date"].notna())]

df = pd.concat([tw_f[shared_cols], iv_f[shared_cols]], ignore_index=True)
df["month"] = df["date"].dt.to_period("M").dt.start_time
df["post_oct7"] = df["date"] >= OCT7

import re
def tokenize(text):
    if pd.isna(text): return []
    t = str(text).lower()
    t = re.sub(r'https?://\S+', '', t)
    t = re.sub(r'[^\w\s\u00e0-\u00ff-]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    stop = {"le","la","les","de","du","des","un","une","et","en","a","au","pas","plus","tout"}
    return [w for w in t.split() if w not in stop and len(w) >= 3]

df["text_tokens"] = df["text_clean"].apply(tokenize)
df_post = df[df["post_oct7"]].copy()

print(f"Total filtré : {len(df):,}")
print(f"Post-7 oct : {len(df_post):,}")

## 2. Fix A1 — Score lexical sans circularité

In [ ]:
from src.pipeline_v4 import compute_lexical_score_fixed

df_post, word_scores_fixed = compute_lexical_score_fixed(df_post)

# Corrélation deputy-level (nouveau score vs stance_v3)
valid = df_post["score_lexical_fixed"].notna() & df_post["stance_v3"].notna()
dep_agg = df_post[valid].groupby("author").agg(
    mean_lex=("score_lexical_fixed", "mean"),
    mean_v3=("stance_v3", "mean"),
    n=("stance_v3", "size"),
).reset_index()
dep_agg = dep_agg[dep_agg["n"] >= 5]

if len(dep_agg) >= 5:
    rho, p = spearmanr(dep_agg["mean_lex"], dep_agg["mean_v3"])
    print(f"Spearman (député-level) score_lexical_fixed vs stance_v3 : rho={rho:.3f}, p={p:.2e}")
    print(f"Si rho > 0.85 : validation tient. Ancien rho (circulaire) ≈ 0.947.")
else:
    print("Pas assez de députés pour corrélation.")

## 3. Fix B4 — Panel fixe et polarisation

In [ ]:
from src.pipeline_v4 import get_panel_fixe, compute_polarization_panel_fixe

df_panel = get_panel_fixe(df_post)
print(f"Députés actifs en continu (≥90% des mois) : {df_panel['author'].nunique()}")
print(f"Textes du panel fixe : {len(df_panel):,}")

month_list, pair_dist, global_pol = compute_polarization_panel_fixe(df_panel)

if len(global_pol) >= 5:
    tau, p = kendalltau(np.arange(len(global_pol)), global_pol)
    print(f"\nKendall tau (polarisation sur panel fixe) : τ={tau:+.3f}, p={p:.4f}")
    print("Si τ > 0 significatif → polarisation réelle. Si τ ≈ 0 → attrition (modérés arrêtent de tweeter).")
else:
    print("Pas assez de mois pour tendance.")

## 4. Pipeline d'annotation v4

In [ ]:
from src.annotation.annotation_v4 import BATCHES, run_annotation_batch_sync

# Définir la clé API (ou utiliser os.environ["OPENAI_API_KEY"])
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    api_key = input("Collez votre clé OpenAI (sk-...) ou Entrée pour SKIP l'annotation : ").strip()
    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key

RUN_ANNOTATION = bool(os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-5-nano"  # ou "gpt-5-mini", "gpt-4o-mini"
print(f"Annotation activée : {RUN_ANNOTATION}")

In [ ]:
# Découpage en batchs et annotation
RETRY_FAILED = True  # True = ré-annoter uniquement les échecs (annotation_failed)

if RUN_ANNOTATION:
    for batch_name, config in BATCHES.items():
        start = pd.Timestamp(config["start"])
        end = pd.Timestamp(config["end"])
        mask = (df_post["date"] >= start) & (df_post["date"] <= end)
        batch_df = df_post[mask].copy().reset_index(drop=True)
        existing_path = OUTPUT_DIR / f"annotations_v4_{batch_name}.parquet"
        if RETRY_FAILED and existing_path.exists():
            existing = pd.read_parquet(existing_path)
            failed_mask = existing.get("annotation_failed", pd.Series([False]*len(existing))).fillna(False)
            n_failed = int(failed_mask.sum())
            if n_failed == 0:
                print(f"Batch {batch_name} : 0 échec, skip.")
                continue
            keep_cols = ["author","group","bloc","date","text_clean","stance_v3","confidence_v3","primary_frame_v3","primary_target_v3","is_off_topic_v3","arena","retweets","likes"]
            batch_df = existing.loc[failed_mask, [c for c in keep_cols if c in existing.columns]].copy().reset_index(drop=True)
            print(f"Batch {batch_name} : {n_failed:,} échecs à ré-annoter")
        elif len(batch_df) == 0:
            print(f"Batch {batch_name} : 0 textes, skip.")
            continue
        else:
            print(f"Batch {batch_name} : {len(batch_df):,} textes")
        result_df = run_annotation_batch_sync(batch_df, batch_name, output_dir=str(OUTPUT_DIR), model=MODEL, concurrency=8)
        if RETRY_FAILED and existing_path.exists():
            success_existing = existing[~failed_mask].copy()
            merged = pd.concat([success_existing, result_df], ignore_index=True)
            merged.to_parquet(existing_path, index=False)
            print(f"  → fusionné")
        else:
            print(f"  → annotations_v4_{batch_name}.parquet sauvegardé.")
else:
    print("Annotation SKIP (pas de clé API). Chargez les parquets v4 existants si disponibles.")

In [ ]:
# Chargement des annotations v4 (si déjà produites)
v4_files = list(OUTPUT_DIR.glob("annotations_v4_*.parquet"))
if v4_files:
    dfs_v4 = []
    for f in v4_files:
        batch_name = f.stem.replace("annotations_v4_", "")
        d = pd.read_parquet(f)
        d["batch"] = batch_name
        dfs_v4.append(d)
    df_v4 = pd.concat(dfs_v4, ignore_index=True)
    print(f"Annotations v4 chargées : {len(df_v4):,} textes")
else:
    df_v4 = pd.DataFrame()
    print("Aucun fichier annotations_v4_*.parquet trouvé.")

## 5. Post-processing et comparaison v3/v4

In [ ]:
if len(df_v4) > 0 and "stance_v4" in df_v4.columns:
    valid = df_v4["stance_v3"].notna() & df_v4["stance_v4"].notna()
    rho, p = spearmanr(df_v4.loc[valid, "stance_v3"], df_v4.loc[valid, "stance_v4"])
    print(f"Corrélation Spearman stance_v3 vs stance_v4 : ρ={rho:.3f}, p={p:.2e}")
    print("Attendu : ρ > 0.7")
    
    # Movers cachés : stance_v3=0 mais ceasefire_call=True
    movers = df_v4[(df_v4["stance_v3"] == 0) & (df_v4["ceasefire_call"] == True)]
    print(f"\nMovers cachés (stance_v3=0, ceasefire_call=True) : {len(movers):,}")
    
    # Taux de flags
    if "flags" in df_v4.columns:
        all_flags = [f for flags in df_v4["flags"].dropna() for f in (flags if isinstance(flags, list) else [])]
        incoherent = sum(1 for f in all_flags if "INCOHERENT" in str(f))
        print(f"Flags INCOHERENT : {incoherent} (seuil <5%)")
else:
    print("Pas de données v4 pour comparaison.")

## 6. Analyses NLP complémentaires

In [ ]:
from src.pipeline_v4 import (
    fightin_words, compute_feel_proportions, has_ceasefire_term,
    plot_ceasefire_lexical_contagion, plot_fightin_words_temporal,
    clean_text, tokenize,
)

# Fightin' Words temporel : CHOC vs RESOL
choc_start, choc_end = pd.Timestamp("2023-10-01"), pd.Timestamp("2024-01-31")
resol_start, resol_end = pd.Timestamp("2024-11-01"), pd.Timestamp("2026-01-31")

df_choc = df_post[(df_post["date"] >= choc_start) & (df_post["date"] <= choc_end)]
df_resol = df_post[(df_post["date"] >= resol_start) & (df_post["date"] <= resol_end)]

gr_choc = df_choc[df_choc["bloc"] == "Gauche radicale"]["text_tokens"].tolist()
dr_choc = df_choc[df_choc["bloc"] == "Droite"]["text_tokens"].tolist()
gr_resol = df_resol[df_resol["bloc"] == "Gauche radicale"]["text_tokens"].tolist()
dr_resol = df_resol[df_resol["bloc"] == "Droite"]["text_tokens"].tolist()

if gr_choc and dr_choc:
    fw_choc = fightin_words(gr_choc, dr_choc)
    print("Fightin' Words CHOC (Gauche rad. vs Droite) — Top 10 mots distinctifs Gauche:")
    print(fw_choc.head(10)[["word", "z_score"]].to_string())
if gr_resol and dr_resol:
    fw_resol = fightin_words(gr_resol, dr_resol)
    print("\nFightin' Words RESOL — Top 10 mots distinctifs Gauche:")
    print(fw_resol.head(10)[["word", "z_score"]].to_string())
    
    # Mots qui apparaissent/disparaissent
    words_choc = set(fw_choc.head(30)["word"])
    words_resol = set(fw_resol.head(30)["word"])
    new_in_resol = words_resol - words_choc
    gone_in_resol = words_choc - words_resol
    print(f"\nNouveaux mots distinctifs en RESOL : {len(new_in_resol)}")
    print(f"Mots disparus en RESOL : {len(gone_in_resol)}")

In [ ]:
# FEEL emotions — proportion anger/fear par bloc × mois
df_feel = compute_feel_proportions(df_post)
print("FEEL (proportions émotions) — échantillon:")
print(df_feel.head(20).to_string())

# Contagion lexicale cessez-le-feu
plot_ceasefire_lexical_contagion(df_post, OUTPUT_DIR / "figures" / "ceasefire_contagion.png")
print("\nFigure : outputs/figures/ceasefire_contagion.png")

## 7. Visualisations de synthèse (post-v4)

In [ ]:
from src.pipeline_v4 import plot_ceasefire_adoption, plot_conditionality_heatmap, plot_emotional_register, plot_event_study, plot_fightin_words_temporal, fightin_words

fig_dir = OUTPUT_DIR / "figures"

if len(df_v4) > 0:
    plot_ceasefire_adoption(df_v4, fig_dir / "v4_ceasefire_adoption.png")
    plot_conditionality_heatmap(df_v4, fig_dir / "v4_conditionality_heatmap.png")
    plot_emotional_register(df_v4, fig_dir / "v4_emotional_register.png")
    plot_event_study(df_v4, fig_dir / "v4_event_study.png")
    print("Figures v4 générées.")
else:
    print("Pas de données v4 pour les figures.")

# Fightin' Words : Gauche rad. (gr vs dr), Droite (dr vs gr)
try:
    fw_choc_gr = fightin_words(gr_choc, dr_choc)
    fw_resol_gr = fightin_words(gr_resol, dr_resol)
    fw_choc_dr = fightin_words(dr_choc, gr_choc)
    fw_resol_dr = fightin_words(dr_resol, gr_resol)
    plot_fightin_words_temporal(fw_choc_gr, fw_resol_gr, "Gauche radicale", fig_dir / "fightin_words_Gauche.png")
    plot_fightin_words_temporal(fw_choc_dr, fw_resol_dr, "Droite", fig_dir / "fightin_words_Droite.png")
    print("Figures Fightin' Words générées.")
except NameError:
    print("Fightin' Words : exécutez d'abord la cellule Analyses NLP.")

In [ ]:
print("Pipeline terminé.")
print("Figures dans:", fig_dir.absolute())